# 🧿 DeepGuard AI — Full Training Pipeline
### EfficientNet-B4 Fine-tuning on 140K Real & Fake Faces
**GPU:** T4 | **Est. time:** 3-5 hours

In [ ]:
# ============================================================
# CELL 1: Install all dependencies
# ============================================================
!pip install timm torch torchvision scikit-learn matplotlib seaborn kaggle huggingface-hub -q
print('✅ All dependencies installed')

In [ ]:
# ============================================================
# CELL 2: Upload kaggle.json and download dataset
# ============================================================
from google.colab import files
import os

print('📁 Upload your kaggle.json file...')
uploaded = files.upload()  # Upload kaggle.json here

# Setup Kaggle credentials
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials set')

# Download 140K Real and Fake Faces dataset
print('⬇️  Downloading dataset (this takes ~5 mins)...')
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data --unzip
print('✅ Dataset downloaded!')

In [ ]:
# ============================================================
# CELL 3: Explore dataset structure
# ============================================================
import os
import glob

# Check structure
for root, dirs, files_list in os.walk('/content/data'):
    level = root.replace('/content/data', '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < 2:
            subindent = ' ' * 2 * (level + 1)
            for f in files_list[:3]:
                print(f'{subindent}{f}')

# Count images
real_imgs = glob.glob('/content/data/**/real/**/*.jpg', recursive=True) + \
            glob.glob('/content/data/**/real/**/*.png', recursive=True)
fake_imgs = glob.glob('/content/data/**/fake/**/*.jpg', recursive=True) + \
            glob.glob('/content/data/**/fake/**/*.png', recursive=True)

print(f'\n✅ Real images: {len(real_imgs)}')
print(f'✅ Fake images: {len(fake_imgs)}')

In [ ]:
# ============================================================
# CELL 4: Build Dataset and DataLoaders
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
import random

class DeepfakeDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
            if self.transform: img = self.transform(img)
            return img, torch.tensor(self.labels[idx], dtype=torch.float32)
        except:
            # Return a random valid sample on error
            return self.__getitem__(random.randint(0, len(self)-1))

# Use up to 30K samples for faster training (increase if you have time)
MAX_PER_CLASS = 15000
real_sample = real_imgs[:MAX_PER_CLASS]
fake_sample = fake_imgs[:MAX_PER_CLASS]

all_paths = real_sample + fake_sample
all_labels = [0]*len(real_sample) + [1]*len(fake_sample)

tr_p, va_p, tr_l, va_l = train_test_split(
    all_paths, all_labels, test_size=0.15, stratify=all_labels, random_state=42
)

train_tf = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((380, 380)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_dl = DataLoader(DeepfakeDataset(tr_p, tr_l, train_tf), batch_size=32, shuffle=True, num_workers=2)
val_dl   = DataLoader(DeepfakeDataset(va_p, va_l, val_tf),   batch_size=32, num_workers=2)

print(f'✅ Train: {len(tr_p)} | Val: {len(va_p)}')
print(f'✅ Batches: {len(train_dl)} train, {len(val_dl)} val')

In [ ]:
# ============================================================
# CELL 5: Build EfficientNet-B4 Model with Attention Head
# ============================================================
import timm

class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b4', pretrained=True, num_classes=0, global_pool='avg')
        in_feat = self.backbone.num_features  # 1792
        self.head = nn.Sequential(
            nn.Linear(in_feat, 512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = DeepfakeDetector().to(DEVICE)
print(f'✅ Device: {DEVICE}')
print(f'✅ Model params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

In [ ]:
# ============================================================
# CELL 6: Train the model
# ============================================================
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

EPOCHS = 15
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.BCEWithLogitsLoss()

best_acc = 0
history = []

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    all_preds, all_labels_list = [], []
    with torch.no_grad():
        for imgs, labels in val_dl:
            probs = torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy()
            all_preds.extend(probs > 0.5)
            all_labels_list.extend(labels.numpy())

    acc = accuracy_score(all_labels_list, all_preds)
    f1  = f1_score(all_labels_list, all_preds)
    scheduler.step()
    history.append({'epoch': epoch+1, 'acc': acc, 'f1': f1})

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Loss: {train_loss/len(train_dl):.4f} | Acc: {acc:.4f} | F1: {f1:.4f}')

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'efficientnet_b4_deepguard.pth')
        print(f'  💾 Best model saved! acc={acc:.4f}')

print(f'\n✅ Training complete! Best accuracy: {best_acc:.4f} ({best_acc*100:.1f}%)')

In [ ]:
# ============================================================
# CELL 7: Plot results
# ============================================================
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
accs   = [h['acc'] for h in history]
f1s    = [h['f1'] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#08090d')
for ax in axes:
    ax.set_facecolor('#0d0f17')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.title.set_color('white')

axes[0].plot(epochs, accs, color='#6366f1', linewidth=2, marker='o')
axes[0].set_title('Validation Accuracy')
axes[0].set_ylim(0.5, 1.0)
axes[0].axhline(y=0.96, color='#10b981', linestyle='--', alpha=0.5, label='96% target')
axes[0].legend()

axes[1].plot(epochs, f1s, color='#8b5cf6', linewidth=2, marker='o')
axes[1].set_title('F1 Score')

plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curve saved')

In [ ]:
# ============================================================
# CELL 8: Download the model weights to your computer
# ============================================================
from google.colab import files

files.download('efficientnet_b4_deepguard.pth')
files.download('training_results.png')

print('✅ Download started!')
print('   Save the .pth file to:')
print('   C:\\Users\\Lenovo\\Desktop\\deepguard-ai\\backend\\models\\efficientnet_b4_deepguard.pth')

In [ ]:
# ============================================================
# CELL 9: Push model to HuggingFace
# ============================================================
# Fill in your HuggingFace token below
HF_TOKEN = 'your_hf_token_here'  # Get from huggingface.co/settings/tokens
HF_REPO  = 'Sowaiba-01/deepguard-ai'

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, repo_type='model', exist_ok=True)
api.upload_file(
    path_or_fileobj='efficientnet_b4_deepguard.pth',
    path_in_repo='efficientnet_b4_deepguard.pth',
    repo_id=HF_REPO,
    repo_type='model',
)
print(f'✅ Model live at: https://huggingface.co/{HF_REPO}')